In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Training 3-Tier Hierarchical LightGBM Pipeline Layers (`models/train_hierarchical_lightgbm_layers.ipynb`)

This notebook trains the **3-Tier 4-Model Hierarchical LightGBM Triage Architecture** using **layer-exclusive feature subsets** and exact class filtering splits:

### 3-Tier 4-Model Layer Feature Subsets & Training Data Splits
1. **Layer 1 LightGBM (ESI 1 Detector)** (`deploy/lightgbm_layer1_esi1_model.rds`):
   - **Features (8 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_hypertension`, `is_tachypnea`, `is_bradypnea`, `is_tachycardia_total`.
   - **Data Split**: Complete Dataset (ESI 1 vs Non-ESI 1).
2. **Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist)** (`deploy/rf_esi23_esi45_extreme_model.rds`):
   - **Features (18 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_dyspnea_moderate`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_max`, `hr_last_to_max`, `sbp_last_to_max`, `rr_last_to_max`.
   - **Data Split**: **Remove ESI 1 rows** (trained on ESI 2, 3, 4, 5).
3. **Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist)** (`deploy/lightgbm_esi23_model.rds`):
   - **Features (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `hr_mean_to_last`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `spo2_mean_to_last`, `rr_mean_to_last`.
   - **Data Split**: **Remove ESI 1, 4, 5 rows** (trained on ESI 2 vs ESI 3).
4. **Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist)** (`deploy/lightgbm_esi45_model.rds`):
   - **Features (9 Features)**: `age`, `gender`, `cc_breathingdifficulty`, `is_bradypnea`, `is_hypotension`, `is_bradycardia_total`, `is_tachycardia_moderate`, `is_dyspnea_total`, `hr_mean_to_last`.
   - **Data Split**: **Remove ESI 1, 2, 3 rows** (trained on ESI 4 vs ESI 5).

### Evaluated Metrics
Reports individual layer performances and evaluates the **Combined 5-Class Soft Probabilistic Model** on the holdout test set.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load & Auto-Install Required R Libraries
# ---------------------------------------------------------
r_pkgs <- c("jsonlite", "dplyr", "ggplot2", "tidyr", "pROC", "lightgbm", "caret")
missing_r <- r_pkgs[!(r_pkgs %in% installed.packages()[, "Package"])]
if (length(missing_r) > 0) {
  install.packages(missing_r, repos = "https://cloud.r-project.org", dependencies = TRUE)
}
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) config_path <- "config/triage_conf.json"
config <- fromJSON(config_path)
set.seed(config$training$random_state)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct Exclusive Layer Feature Sets
# ---------------------------------------------------------
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) data_file <- paste0("../", data_file)
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
pulse_last <- get_vec("pulse_last"); pulse_max <- get_vec("pulse_max"); pulse_min <- get_vec("pulse_min")
sbp_last   <- get_vec("sbp_last");   sbp_max   <- get_vec("sbp_max");   sbp_min   <- get_vec("sbp_min")
spo2_last  <- get_vec("spo2_last");  spo2_max  <- get_vec("spo2_max");  spo2_min  <- get_vec("spo2_min")
resp_last  <- get_vec("resp_last");  resp_max  <- get_vec("resp_max");  resp_min  <- get_vec("resp_min")
t_hr       <- get_vec("triage_vital_hr"); t_sbp <- get_vec("triage_vital_sbp"); t_o2 <- get_vec("triage_vital_o2"); t_rr <- get_vec("triage_vital_rr")
# Construct Master Dataframe with Unique Columns
df_master <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  # Clinical Binary Features
  is_dyspnea_total        = ifelse(t_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(t_o2 > 90 & t_o2 < 94, 1, 0),
  is_hypertension         = ifelse(t_sbp > 220, 1, 0),
  is_hypotension          = ifelse(t_sbp <= 90, 1, 0),
  is_tachypnea            = ifelse(t_rr > 30, 1, 0),
  is_bradypnea            = ifelse(t_rr < 10, 1, 0),
  is_tachycardia_total    = ifelse(t_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(t_hr > 100 & t_hr < 150, 1, 0),
  is_bradycardia_total    = ifelse(t_hr < 40, 1, 0),
  # Vital Deltas & Ranges
  hr_mean_to_last         = t_hr - pulse_last,
  spo2_mean_to_last       = t_o2 - spo2_last,
  rr_mean_to_last         = t_rr - resp_last,
  hr_range                = pulse_max - pulse_min,
  rr_range                = resp_max - resp_min,
  spo2_range              = spo2_max - spo2_min,
  sbp_range               = sbp_max - sbp_min,
  hr_last_to_min          = pulse_last - pulse_min,
  rr_last_to_min          = resp_last - resp_min,
  spo2_last_to_max        = spo2_last - spo2_max,
  hr_last_to_max          = pulse_last - pulse_max,
  sbp_last_to_max         = sbp_last - sbp_max,
  rr_last_to_max          = resp_last - resp_max
)
# Define Layer-Exclusive Feature Lists
l1_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_hypertension", "is_tachypnea", "is_bradypnea", "is_tachycardia_total")
l2_feat_names  <- c("age", "gender", "cc_breathingdifficulty", "is_dyspnea_moderate", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")
l3a_feat_names <- c("age", "gender", "cc_breathingdifficulty", "hr_mean_to_last", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "spo2_mean_to_last", "rr_mean_to_last")
l3b_feat_names <- c("age", "gender", "cc_breathingdifficulty", "is_bradypnea", "is_hypotension", "is_bradycardia_total", "is_tachycardia_moderate", "is_dyspnea_total", "hr_mean_to_last")
raw_esi <- as.character(raw_df[[target_col_name]])
df_master$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
df_master <- na.omit(df_master)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_master$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_master[in_train_val, ]
test_df      <- df_master[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# Continuous features to standard scale
cont_cols <- c("age", "hr_mean_to_last", "spo2_mean_to_last", "rr_mean_to_last", "hr_range", "rr_range", "spo2_range", "sbp_range", "hr_last_to_min", "rr_last_to_min", "spo2_last_to_max", "hr_last_to_max", "sbp_last_to_max", "rr_last_to_max")
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_scaled <- train_df
val_scaled   <- val_df
test_scaled  <- test_df
train_scaled[, cont_cols] <- predict(preproc, train_df[, cont_cols])
val_scaled[, cont_cols]   <- predict(preproc, val_df[, cont_cols])
test_scaled[, cont_cols]  <- predict(preproc, test_df[, cont_cols])
cat(sprintf("Partitions Prepared: Train=%d, Val=%d, Test=%d\n", nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Train Layer 1 LightGBM (ESI 1 Detector - 8 Features)
# ---------------------------------------------------------
set.seed(config$training$random_state)
y_tr_l1 <- ifelse(train_scaled$target_col == "1", 1, 0)
y_vl_l1 <- ifelse(val_scaled$target_col == "1", 1, 0)
y_ts_l1 <- ifelse(test_scaled$target_col == "1", 1, 0)
dtrain_l1 <- lgb.Dataset(data = as.matrix(train_scaled[, l1_feat_names]), label = y_tr_l1)
dval_l1   <- lgb.Dataset(data = as.matrix(val_scaled[, l1_feat_names]),   label = y_vl_l1)
params_l1 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l1_model <- lgb.train(
  params                = params_l1,
  data                  = dtrain_l1,
  nrounds               = 100,
  valids                = list(train = dtrain_l1, val = dval_l1),
  early_stopping_rounds = 10,
  verbose               = 0
)
# Calibrate Layer 1 Threshold via 99th Percentile on Val Split
p_l1_val <- predict(lgb_l1_model, as.matrix(val_scaled[, l1_feat_names]))
l1_threshold <- quantile(p_l1_val, 0.99, na.rm = TRUE)
p_l1_test <- predict(lgb_l1_model, as.matrix(test_scaled[, l1_feat_names]))
auc_l1_test <- as.numeric(pROC::roc(y_ts_l1, p_l1_test)$auc)
cm_l1 <- confusionMatrix(factor(ifelse(p_l1_test >= l1_threshold, 1, 0), levels = c(1, 0)), factor(y_ts_l1, levels = c(1, 0)))
rec_l1  <- cm_l1$byClass["Sensitivity"]
spec_l1 <- cm_l1$byClass["Specificity"]
bal_l1  <- cm_l1$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat(sprintf("   LAYER 1 LIGHTGBM (ESI 1 DETECTOR - 8 FEATS, THRESHOLD = %.4f) TEST REPORT\n", l1_threshold))
cat("============================================================\n")
cat(sprintf("  ESI 1 Sensitivity (Recall) : %.4f\n", rec_l1))
cat(sprintf("  Non-ESI 1 Specificity      : %.4f\n", spec_l1))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l1))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l1_test))
cat("============================================================\n\n")
print(cm_l1$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Layer 2 LightGBM (ESI 2/3 vs ESI 4/5 Specialist - 18 Features - ESI 1 Removed)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1 ROWS FOR LAYER 2
train_l2 <- train_scaled %>% filter(target_col != "1")
val_l2   <- val_scaled   %>% filter(target_col != "1")
test_l2  <- test_scaled  %>% filter(target_col != "1")
y_tr_l2 <- ifelse(train_l2$target_col %in% c("2", "3"), 1, 0) # 1 = ESI 2/3, 0 = ESI 4/5
y_vl_l2 <- ifelse(val_l2$target_col %in% c("2", "3"), 1, 0)
y_ts_l2 <- ifelse(test_l2$target_col %in% c("2", "3"), 1, 0)
dtrain_l2 <- lgb.Dataset(data = as.matrix(train_l2[, l2_feat_names]), label = y_tr_l2)
dval_l2   <- lgb.Dataset(data = as.matrix(val_l2[, l2_feat_names]),   label = y_vl_l2)
params_l2 <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l2_model <- lgb.train(
  params                = params_l2,
  data                  = dtrain_l2,
  nrounds               = 100,
  valids                = list(train = dtrain_l2, val = dval_l2),
  early_stopping_rounds = 10,
  verbose               = 0
)
p_l2_test <- predict(lgb_l2_model, as.matrix(test_l2[, l2_feat_names]))
auc_l2_test <- as.numeric(pROC::roc(y_ts_l2, p_l2_test)$auc)
cm_l2 <- confusionMatrix(factor(ifelse(p_l2_test >= 0.5, 1, 0), levels = c(1, 0)), factor(y_ts_l2, levels = c(1, 0)))
rec_l2  <- cm_l2$byClass["Sensitivity"]
spec_l2 <- cm_l2$byClass["Specificity"]
bal_l2  <- cm_l2$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat("   LAYER 2 LIGHTGBM (ESI 2/3 vs 4/5 SPECIALIST - 18 FEATS) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 2/3 Recall (Sens)      : %.4f\n", rec_l2))
cat(sprintf("  ESI 4/5 Specificity        : %.4f\n", spec_l2))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l2))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l2_test))
cat("============================================================\n\n")
print(cm_l2$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Train Layer 3A LightGBM (ESI 2 vs ESI 3 Specialist - 9 Features - ESI 1,4,5 Removed)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1, 4, 5 ROWS FOR LAYER 3A
train_l3a <- train_scaled %>% filter(target_col %in% c("2", "3"))
val_l3a   <- val_scaled   %>% filter(target_col %in% c("2", "3"))
test_l3a  <- test_scaled  %>% filter(target_col %in% c("2", "3"))
y_tr_l3a <- ifelse(train_l3a$target_col == "2", 1, 0) # 1 = ESI 2, 0 = ESI 3
y_vl_l3a <- ifelse(val_l3a$target_col == "2", 1, 0)
y_ts_l3a <- ifelse(test_l3a$target_col == "2", 1, 0)
dtrain_l3a <- lgb.Dataset(data = as.matrix(train_l3a[, l3a_feat_names]), label = y_tr_l3a)
dval_l3a   <- lgb.Dataset(data = as.matrix(val_l3a[, l3a_feat_names]),   label = y_vl_l3a)
params_l3a <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l3a_model <- lgb.train(
  params                = params_l3a,
  data                  = dtrain_l3a,
  nrounds               = 100,
  valids                = list(train = dtrain_l3a, val = dval_l3a),
  early_stopping_rounds = 10,
  verbose               = 0
)
p_l3a_test <- predict(lgb_l3a_model, as.matrix(test_l3a[, l3a_feat_names]))
auc_l3a_test <- as.numeric(pROC::roc(y_ts_l3a, p_l3a_test)$auc)
cm_l3a <- confusionMatrix(factor(ifelse(p_l3a_test >= 0.5, 1, 0), levels = c(1, 0)), factor(y_ts_l3a, levels = c(1, 0)))
rec_l3a  <- cm_l3a$byClass["Sensitivity"]
spec_l3a <- cm_l3a$byClass["Specificity"]
bal_l3a  <- cm_l3a$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat("   LAYER 3A LIGHTGBM (ESI 2 vs ESI 3 SPECIALIST - 9 FEATS) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 2 Sensitivity (Recall) : %.4f\n", rec_l3a))
cat(sprintf("  ESI 3 Specificity          : %.4f\n", spec_l3a))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l3a))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l3a_test))
cat("============================================================\n\n")
print(cm_l3a$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Train Layer 3B LightGBM (ESI 4 vs ESI 5 Specialist - 9 Features - ESI 1,2,3 Removed)
# ---------------------------------------------------------
set.seed(config$training$random_state)
# REMOVE ESI 1, 2, 3 ROWS FOR LAYER 3B
train_l3b <- train_scaled %>% filter(target_col %in% c("4", "5"))
val_l3b   <- val_scaled   %>% filter(target_col %in% c("4", "5"))
test_l3b  <- test_scaled  %>% filter(target_col %in% c("4", "5"))
y_tr_l3b <- ifelse(train_l3b$target_col == "4", 1, 0) # 1 = ESI 4, 0 = ESI 5
y_vl_l3b <- ifelse(val_l3b$target_col == "4", 1, 0)
y_ts_l3b <- ifelse(test_l3b$target_col == "4", 1, 0)
dtrain_l3b <- lgb.Dataset(data = as.matrix(train_l3b[, l3b_feat_names]), label = y_tr_l3b)
dval_l3b   <- lgb.Dataset(data = as.matrix(val_l3b[, l3b_feat_names]),   label = y_vl_l3b)
params_l3b <- list(
  objective        = "binary",
  metric           = "binary_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  max_depth        = 6,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1
)
lgb_l3b_model <- lgb.train(
  params                = params_l3b,
  data                  = dtrain_l3b,
  nrounds               = 100,
  valids                = list(train = dtrain_l3b, val = dval_l3b),
  early_stopping_rounds = 10,
  verbose               = 0
)
p_l3b_test <- predict(lgb_l3b_model, as.matrix(test_l3b[, l3b_feat_names]))
auc_l3b_test <- as.numeric(pROC::roc(y_ts_l3b, p_l3b_test)$auc)
cm_l3b <- confusionMatrix(factor(ifelse(p_l3b_test >= 0.5, 1, 0), levels = c(1, 0)), factor(y_ts_l3b, levels = c(1, 0)))
rec_l3b  <- cm_l3b$byClass["Sensitivity"]
spec_l3b <- cm_l3b$byClass["Specificity"]
bal_l3b  <- cm_l3b$byClass["Balanced Accuracy"]
cat("============================================================\n")
cat("   LAYER 3B LIGHTGBM (ESI 4 vs ESI 5 SPECIALIST - 9 FEATS) TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  ESI 4 Sensitivity (Recall) : %.4f\n", rec_l3b))
cat(sprintf("  ESI 5 Specificity          : %.4f\n", spec_l3b))
cat(sprintf("  Balanced Accuracy          : %.4f\n", bal_l3b))
cat(sprintf("  ROC-AUC (Test)             : %.4f\n", auc_l3b_test))
cat("============================================================\n\n")
print(cm_l3b$table)
cat("\n\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save All 4 Model Artifacts to deploy/
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = lgb_l1_model,  preproc = preproc, is_l1_lgb  = TRUE), file = file.path(deploy_dir, "lightgbm_layer1_esi1_model.rds"))
saveRDS(list(model = lgb_l2_model,  preproc = preproc, is_lgb_l2  = TRUE), file = file.path(deploy_dir, "rf_esi23_esi45_extreme_model.rds"))
saveRDS(list(model = lgb_l3a_model, preproc = preproc, is_lgb_l3a = TRUE), file = file.path(deploy_dir, "lightgbm_esi23_model.rds"))
saveRDS(list(model = lgb_l3b_model, preproc = preproc, is_lgb_l3b = TRUE), file = file.path(deploy_dir, "lightgbm_esi45_model.rds"))
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
l1_l4_report <- data.frame(
  Layer = c("Layer1_ESI1_Detector", "Layer2_ESI23_vs_ESI45", "Layer3A_ESI2_vs_ESI3", "Layer3B_ESI4_vs_ESI5"),
  Input_Features = c("8_Features", "18_Features", "9_Features", "9_Features"),
  Recall = c(round(rec_l1, 4), round(rec_l2, 4), round(rec_l3a, 4), round(rec_l3b, 4)),
  Specificity = c(round(spec_l1, 4), round(spec_l2, 4), round(spec_l3a, 4), round(spec_l3b, 4)),
  Balanced_Accuracy = c(round(bal_l1, 4), round(bal_l2, 4), round(bal_l3a, 4), round(bal_l3b, 4)),
  ROC_AUC = c(round(auc_l1_test, 4), round(auc_l2_test, 4), round(auc_l3a_test, 4), round(auc_l3b_test, 4))
)
write.csv(l1_l4_report, file = file.path(reports_dir, "hierarchical_l1_l2_training_report.csv"), row.names = FALSE)
cat("All 4 Sub-Model Artifacts saved to deploy/ & Report saved to reports/hierarchical_l1_l2_training_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 8: Evaluate Combined 5-Class Soft Probabilistic Pipeline
# ---------------------------------------------------------
X_test_l1  <- as.matrix(test_scaled[, l1_feat_names])
X_test_l2  <- as.matrix(test_scaled[, l2_feat_names])
X_test_l3a <- as.matrix(test_scaled[, l3a_feat_names])
X_test_l3b <- as.matrix(test_scaled[, l3b_feat_names])
p1_test  <- predict(lgb_l1_model,  X_test_l1)
p2_test  <- predict(lgb_l2_model,  X_test_l2)
p3a_test <- predict(lgb_l3a_model, X_test_l3a)
p3b_test <- predict(lgb_l3b_model, X_test_l3b)
probs_comb <- matrix(0, nrow = nrow(test_df), ncol = 5)
colnames(probs_comb) <- c("1", "2", "3", "4", "5")
probs_comb[, 1] <- p1_test
probs_comb[, 2] <- (1 - p1_test) * p2_test * p3a_test
probs_comb[, 3] <- (1 - p1_test) * p2_test * (1 - p3a_test)
probs_comb[, 4] <- (1 - p1_test) * (1 - p2_test) * p3b_test
probs_comb[, 5] <- (1 - p1_test) * (1 - p2_test) * (1 - p3b_test)
preds_comb <- factor(apply(probs_comb, 1, which.max), levels = 1:5)
act_comb   <- factor(as.numeric(as.character(test_df$target_col)), levels = 1:5)
cm_comb <- confusionMatrix(preds_comb, act_comb)
rec_comb_list  <- as.numeric(cm_comb$byClass[, "Sensitivity"])
spec_comb_list <- as.numeric(cm_comb$byClass[, "Specificity"])
bal_comb_list  <- as.numeric(cm_comb$byClass[, "Balanced Accuracy"])
rec_comb_list[is.na(rec_comb_list)]   <- 0
spec_comb_list[is.na(spec_comb_list)] <- 0
bal_comb_list[is.na(bal_comb_list)]   <- 0
auc_comb_list <- sapply(1:5, function(i) {
  act_bin <- ifelse(act_comb == i, 1, 0)
  r_obj <- tryCatch(pROC::roc(act_bin, probs_comb[, i]), error = function(e) NULL)
  if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
})
macro_rec_comb  <- mean(rec_comb_list)
macro_spec_comb <- mean(spec_comb_list)
macro_bal_comb  <- mean(bal_comb_list)
macro_auc_comb  <- mean(auc_comb_list, na.rm = TRUE)
cat("============================================================\n")
cat("   COMBINED 5-CLASS SOFT PROBABILISTIC MODEL TEST REPORT\n")
cat("============================================================\n")
cat(sprintf("  Macro Recall (Sensitivity) : %.4f\n", macro_rec_comb))
cat(sprintf("  Macro Specificity          : %.4f\n", macro_spec_comb))
cat(sprintf("  Macro Balanced Accuracy    : %.4f\n", macro_bal_comb))
cat(sprintf("  Macro ROC-AUC              : %.4f\n", macro_auc_comb))
cat("============================================================\n\n")
cat("Confusion Matrix (Combined 5-Class Soft Probabilistic Model):\n")
print(cm_comb$table)
cat("\n\n")
# Save Combined Soft Probability Model Report
soft_comb_report <- data.frame(
  Pipeline = "Combined_5Class_Soft_Probability_Model",
  Macro_Recall = round(macro_rec_comb, 4),
  Macro_Specificity = round(macro_spec_comb, 4),
  Macro_Balanced_Accuracy = round(macro_bal_comb, 4),
  Macro_ROC_AUC = round(macro_auc_comb, 4)
)
write.csv(soft_comb_report, file = file.path(reports_dir, "combined_soft_pipeline_training_report.csv"), row.names = FALSE)
cat("Combined Soft Probability Model Training Report saved to reports/combined_soft_pipeline_training_report.csv\n")